# MIST NPLIB1 tests

MIST retrieval accuracy on the NPLIB1 test split with the fingerprint vs contrastive method

In [ ]:
from conformal.mist import (
    fingerprint,
    load_contrastive,
    load_fingerprint,
    dict_to,
    spectra_loader,
)
from conformal.nplib1 import load_split, load_spectrum

In [2]:
import os

os.chdir("mist")

### Fingerprints

In [3]:
model, kwargs = load_fingerprint()

In [4]:
from pathlib import Path

test = load_split("test", Path("../NPLIB1"))

len(test)

819

In [ ]:
from tqdm import tqdm
from mist.data import featurizers
import numpy as np
import torch

device = "cuda"

model.to(device)
model.eval()
spec_featurizer = featurizers.get_spec_featurizer(**kwargs)

ranks: list[int] = []

for entry in tqdm(test):
    try:
        truth_index = entry["candidates"].index(entry["truth"])
    except ValueError:
        print("truth not in candidates")
        continue

    # 1. Compute spectrum fingerprint
    spectrum = load_spectrum(entry["spectrum"], Path("../NPLIB1"))
    loader = spectra_loader([spectrum], spec_featurizer, batch_size=1)
    batch = next(iter(loader))  # only one spectrum

    with torch.no_grad():
        # (1, 4096)
        fp, _ = model.encode_spectra(dict_to(batch, device))

    # 2. Compute candidate fingerprints
    # (n, 4096)
    fingerprints = np.stack(
        [fingerprint(smile) for smile in entry["candidates"]], dtype=np.float64
    )
    fingerprints /= np.linalg.norm(fingerprints, ord=2, axis=-1)[:, None]
    fp = fp.cpu().numpy()[0]
    fp /= np.linalg.norm(fp, ord=2)

    similarities = fingerprints @ fp

    all_ranks = (-similarities).argsort().argsort()
    ranks.append(all_ranks[truth_index])

len(ranks)

In [ ]:
ks = [1, 5, 20]

for k in ks:
    accuracy = (np.array(ranks) < k).mean()
    print(f"top {k} accuracy: {100 * accuracy:.2f}%")

top 1 accuracy: 26.37%
top 5 accuracy: 52.99%
top 20 accuracy: 70.70%


### Contrastive embeddings

In [43]:
contrast_model, contrast_kwargs = load_contrastive()

In [ ]:
from torch import Tensor
from torch.utils.data import DataLoader
import torch.nn.functional as F

device = "cuda"
contrast_model.to(device)
contrast_model.eval()

ranks: list[int] = []
spec_featurizer = featurizers.get_spec_featurizer(**contrast_kwargs)


for entry in tqdm(test):
    try:
        truth_index = entry["candidates"].index(entry["truth"])
    except ValueError:
        print("truth not in candidates")
        continue

    # 1. Embed spectrum
    spectrum = load_spectrum(entry["spectrum"], Path("../NPLIB1"))
    loader = spectra_loader([spectrum], spec_featurizer, batch_size=1)
    batch = next(iter(loader))  # only one spectrum

    with torch.no_grad():
        _, spec = contrast_model.encode_spectra(dict_to(batch, device))
    # (1, 256)
    spec_embed = spec["contrast"].cpu()

    # 2. Embed candidates
    # (n, 4096)
    fingerprints = np.stack(
        [fingerprint(smile) for smile in entry["candidates"]], dtype=np.float64
    )
    loader = DataLoader(fingerprints, batch_size=64)

    cand_embeds: list[Tensor] = []
    for fps in loader:
        with torch.no_grad():
            _, embeds = contrast_model.encode_mol({"mols": fps.to(device)})
        cand_embeds.append(embeds["contrast"].cpu())
    all_embeds = torch.cat(cand_embeds, dim=0)

    all_embeds = F.normalize(all_embeds, p=2, dim=-1)
    spec_embed = F.normalize(spec_embed, p=2, dim=-1)[0]

    similarities = all_embeds @ spec_embed

    all_ranks = (-similarities).argsort().argsort()
    ranks.append(int(all_ranks[truth_index].item()))


In [56]:
ks = [1, 5, 20]

for k in ks:
    accuracy = (np.array(ranks) < k).mean()
    print(f"top {k} accuracy: {100 * accuracy:.2f}%")

top 1 accuracy: 27.72%
top 5 accuracy: 61.42%
top 20 accuracy: 80.34%
